# tp-13 戦略探索 レベル 0: モデルの順位だけで買う規則ベースの全券種（オッズを使わない）

**目的**: 回収率 100% 超の戦略を探し続けるための土台。最初に、**オッズを一切使わず**、モデルの確率の順位だけで買い目を決める規則を全券種で試す。買い目が確定オッズに依存しないので、確定の払戻表で精算しても先読みにならず、バックテストが偏らない。

## 試すもの
- 券種: 単勝・複勝・馬連・ワイド・馬単・3 連複・3 連単（枠連は枠番の対応が要るので後回し）。
- 買い方: 1 点、上位 k 頭の箱、軸 1 頭の流し（相手 m 頭）、3 連単のフォーメーション。順位は勝率順と複勝率順。
- レースの絞り込み: 最有力馬の勝率が 0 / 0.2 / 0.3 / 0.4 以上のレースだけ。
- 比較: 同じ構造で馬を**ランダム**に選んだ基準（モデルの寄与 = 差）、控除後の払戻率（単勝・複勝 80%、馬連・ワイド 77.5%、馬単・3 連複 75%、3 連単 72.5%）。
- **多重検定**: 試した全戦略に White の reality check（レース単位 bootstrap）をかけ、『最良が 100% を超えて見えるのは偶然を超えるか』を p 値で出す。

## 評価の手順
- 選択は valid 2022〜2024（OOF）だけで行う。上位を選んだら、test 2025 は一度だけ評価する（選んだ数の分だけ多重検定を補正）。
- 戦略の数は記録しておく（補正のため）。購入額の段階づけ（見送り / 500 / 1000 / 2000 円）は、オッズ情報を使う戦略（tp-11/12）で扱う。ここは各戦略を『1 点 100 円』の等額で比べる。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, time
from datetime import datetime
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.betting import lab
from src.pipeline.mlflow.train_utils import require_tracking_server

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
BET = REPO_ROOT / "data/local/modeling/betting"
TAG = os.environ.get("TP13_TAG", "lab0_v1")
OUT = BET / "lab" / TAG
OUT.mkdir(parents=True, exist_ok=True)
LOCK = BET / "lab" / f"TEST2025_EVALUATED_{TAG}.json"
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
mlflow.set_experiment("keiba-tp-betting-lab")
require_tracking_server()
YEARS = ["2022", "2023", "2024", "2025"]
wp = pd.read_parquet(BET / "odds_win_place.parquet"); wp = wp[wp["race_id"].str[:4].isin(YEARS) & (wp["win_odds"] > 0)]
pay_df = pd.read_parquet(BET / "payoff.parquet"); pay_df = pay_df[pay_df["race_id"].str[:4].isin(YEARS)]
cols = ["race_id", "horse_id", "year", "p_win", "p_top2", "p_top3"]
pred = pd.concat([pd.read_parquet(WORK / "oof/model_oof/ensemble_v2.parquet")[cols], pd.read_parquet(WORK / "final_eval_v2/test2025_predictions.parquet")[cols]], ignore_index=True)
pred = pred[pred["year"].between(2022, 2025)]
ORD = lab.build_orders(pred, wp[["race_id", "horse_id", "horse_number"]].drop_duplicates(["race_id", "horse_id"]))
PAY = lab.payoff_lookup(pay_df)
RACES = {y: sorted(r for r in ORD if r[:4] == y) for y in YEARS}
print({y: len(v) for y, v in RACES.items()}, "| 払戻のあるレース", len(PAY))

In [ ]:
# 規則の一覧（戦略 = 規則 × 絞り込み）
RULES = []
for o in ("win", "top3"):
    RULES += [lab.Rule("fukusho", "single", order=o), lab.Rule("fukusho", "box", 2, order=o), lab.Rule("fukusho", "box", 3, order=o)]
    for k in (2, 3, 4, 5):
        RULES += [lab.Rule("wide", "box", k, order=o), lab.Rule("sanrenpuku", "box", max(k, 3) + (1 if k > 3 else 0), order=o)] if k >= 3 else [lab.Rule("wide", "box", k, order=o)]
    for m in (2, 3, 4, 5, 6):
        RULES += [lab.Rule("wide", "nagashi", m, order=o), lab.Rule("sanrenpuku", "nagashi", m, order=o)]
RULES += [lab.Rule("tansho", "single")]
for k in (2, 3, 4, 5):
    RULES += [lab.Rule("tansho", "box", k), lab.Rule("umaren", "box", k)]
for k in (2, 3, 4):
    RULES += [lab.Rule("umatan", "box", k)]
for m in (2, 3, 4, 5, 6):
    RULES += [lab.Rule("umaren", "nagashi", m), lab.Rule("umatan", "nagashi", m)]
for k in (3, 4, 5, 6):
    RULES += [lab.Rule("sanrenpuku", "box", k)]
for k in (3, 4, 5):
    RULES += [lab.Rule("sanrentan", "box", k)]
for m in (3, 4, 5, 6):
    RULES += [lab.Rule("sanrentan", "nagashi", m)]
for a, b, c in ((1, 2, 3), (1, 2, 4), (1, 3, 5), (2, 3, 5), (2, 4, 6)):
    RULES += [lab.Rule("sanrentan", "formation", a, b, c)]
RULES = sorted(set(RULES), key=lambda r: r.name())
FILTERS = (0.0, 0.2, 0.3, 0.4)
STRATS = [(r, f) for r in RULES for f in FILTERS]
print(f"規則 {len(RULES)} × 絞り込み {len(FILTERS)} = 戦略 {len(STRATS)} 個")

In [ ]:
t0 = time.time()
PER = {}                         # (戦略, 年) → レースごとの stake・ret
BASE = {}
for si, (r, f) in enumerate(STRATS):
    for y in YEARS[:3]:
        PER[(si, y)] = lab.evaluate_rule(r, ORD, PAY, RACES[y], min_p1=f)
        BASE[(si, y)] = pd.concat([lab.evaluate_rule(r, ORD, PAY, RACES[y], random_seed=s, min_p1=0.0) for s in range(2)])
print(f"valid 2022〜2024 の評価 {time.time() - t0:.0f}s")
rows = []
for si, (r, f) in enumerate(STRATS):
    d = pd.concat([PER[(si, y)] for y in YEARS[:3]])
    b = pd.concat([BASE[(si, y)] for y in YEARS[:3]])
    if len(d) < 500:
        continue
    ci = lab.roi_ci(d, n_boot=300)
    rows.append({"si": si, "戦略": r.name(), "絞り込み(最有力の勝率>=)": f, **ci, "ランダム基準の回収率": float(b["ret"].sum() / max(b["stake"].sum(), 1e-12)),
                 **{f"roi_{y}": float(PER[(si, y)]["ret"].sum() / max(PER[(si, y)]["stake"].sum(), 1e-12)) if len(PER[(si, y)]) else np.nan for y in YEARS[:3]}})
V = pd.DataFrame(rows).set_index("si")
V["モデルの寄与(回収率−ランダム)"] = V["roi"] - V["ランダム基準の回収率"]
V.to_csv(OUT / "valid_all_strategies.csv")
display(V.sort_values("roi", ascending=False).head(15).round(3)[["戦略", "絞り込み(最有力の勝率>=)", "roi", "lo", "hi", "races", "stake_per_race", "ランダム基準の回収率", "roi_2022", "roi_2023", "roi_2024"]])
print("券種ごとの回収率の分布（valid 2022〜2024、全戦略）:")
V["券種"] = V["戦略"].str.split(":").str[0]
display(V.groupby("券種")["roi"].describe().round(3))

## 多重検定（reality check）
試した戦略すべてについて、レースごとの損益（円 / 1000）を並べ、『期待損益が 0 以下の戦略ばかり』という帰無仮説のもとで、最良の戦略の平均損益が偶然を超えるかを見る。

In [ ]:
all_races = sorted(r for y in YEARS[:3] for r in RACES[y])
idx = {r: i for i, r in enumerate(all_races)}
cols_ = [si for si in V.index]
M = np.zeros((len(all_races), len(cols_)))
for j, si in enumerate(cols_):
    for y in YEARS[:3]:
        d = PER[(si, y)]
        M[[idx[r] for r in d["race_id"]], j] = ((d["ret"] - d["stake"]) / 1000.0).to_numpy()
rc = lab.reality_check(M, n_boot=1000)
best_si = cols_[rc["best"]]
print(f"試した戦略 {rc['n_strategies']} 個。標準化した最良 = {V.loc[best_si, '戦略']}（絞り込み {V.loc[best_si, '絞り込み(最有力の勝率>=)']}）、平均損益 {rc['best_mean'] * 1000:.1f} 円/レース、reality check の p 値 = {rc['p_value']:.3f}")
mean_profit = pd.Series(M.mean(0) * 1000, index=cols_)
print("平均損益が正の戦略数:", int((mean_profit > 0).sum()), "/", len(cols_))
json.dump({"n_strategies": rc["n_strategies"], "best": V.loc[best_si, "戦略"], "best_filter": float(V.loc[best_si, "絞り込み(最有力の勝率>=)"]), "best_mean_yen_per_race": rc["best_mean"] * 1000, "p_value": rc["p_value"],
           "n_positive": int((mean_profit > 0).sum())}, open(OUT / "reality_check_valid.json", "w"), ensure_ascii=False, indent=1)

## test 2025（一度だけ）
valid の上位 5 戦略（賭け金が極端に少ない戦略を除く）を 2025 年に適用する。5 戦略の中の最良に対して、reality check（戦略数 5）の p 値も出す。

In [ ]:
if LOCK.exists() and os.environ.get("TP13_FORCE") != "1":
    raise SystemExit(f"test 2025 は評価済みです（{LOCK}）。結果を見てやり直さない")
cand = V[V["races"] >= 1500].sort_values("roi", ascending=False).head(5)
rows, PM = [], []
for si in cand.index:
    r, f = STRATS[si]
    d = lab.evaluate_rule(r, ORD, PAY, RACES["2025"], min_p1=f)
    b = pd.concat([lab.evaluate_rule(r, ORD, PAY, RACES["2025"], random_seed=s) for s in range(3)])
    ci = lab.roi_ci(d, n_boot=1000)
    rows.append({"戦略": r.name(), "絞り込み": f, "valid の回収率": float(cand.loc[si, "roi"]), "test 2025 の回収率": ci["roi"], "lo": ci["lo"], "hi": ci["hi"], "races": ci["races"], "ランダム基準": float(b["ret"].sum() / b["stake"].sum())})
    full = np.zeros(len(RACES["2025"])); ix = {x: i for i, x in enumerate(RACES["2025"])}
    full[[ix[x] for x in d["race_id"]]] = ((d["ret"] - d["stake"]) / 1000.0).to_numpy(); PM.append(full)
T = pd.DataFrame(rows)
display(T.round(3)); T.to_csv(OUT / "test2025_top5.csv", index=False)
rc5 = lab.reality_check(np.column_stack(PM), n_boot=1000)
print("test 2025: 5 戦略の最良に対する reality check の p 値 =", round(rc5["p_value"], 3))
res = {"evaluated_at": datetime.now().isoformat(timespec="seconds"), "tag": TAG, "git_commit": GIT_COMMIT, "n_strategies": len(STRATS), "valid_reality_check": json.load(open(OUT / "reality_check_valid.json")), "test2025": T.to_dict("records"), "test2025_p": rc5["p_value"]}
(OUT / "report.json").write_text(json.dumps(res, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
with mlflow.start_run(run_name=f"tp-13-lab0-{TAG}", tags={"notebook": "tp-13", "git_commit": GIT_COMMIT, "test_year": "2025"}):
    mlflow.log_metrics({"n_strategies": len(STRATS), "valid_rc_p": float(res["valid_reality_check"]["p_value"]), "test2025_rc_p": float(rc5["p_value"])})
    for i_, r_ in T.iterrows():
        mlflow.log_metrics({f"test2025_roi__{i_:02d}": float(r_["test 2025 の回収率"])})
    for p in OUT.glob("*"):
        mlflow.log_artifact(str(p), "lab0")
LOCK.write_text(json.dumps({"evaluated_at": res["evaluated_at"], "git_commit": GIT_COMMIT, "note": "test 2025 のレベル0評価は実施済み。結果を見てやり直さない"}, ensure_ascii=False))
print("完了")